# M3 · 少算 —— Turbo 8 步 vs 基线 40 步

**学习目标：** 在你的 W7900 上测量*少算*候选（Turbo 蒸馏 DiT、8 步调度）与 B0 基线的差距，把实测加速比与 Amdahl 预测对比，并诚实分析**混淆变量**：B0→L1 同时改变了模型变体和采样调度，并不是只改步数这一个旋钮。

**先预测：** 朴素步数比 40/8 = 5×。你会测到 5× 吗？写下你的预测与理由。

参考讲义：[M03 讲义](../course/M03_less_computation/LESSON.zh-CN.md) · [English](./03_less_computation.ipynb)

In [ ]:
from pathlib import Path
import json
import statistics
import sys
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "lab/reference_runs.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open Jupyter from the course repository root (see notebooks/README.md).")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
COURSE_MODE = "REFERENCE"  # <- edit once: REFERENCE | GPU_CLASSROOM | GPU_RESEARCH
RUN_OWN_GPU_ARMS = True   # <- False in GPU modes: reuse the latest physical run (shown run_id)
from lab import notebook_runtime as rt
from lab import notebook_support as ns
session = rt.begin(COURSE_MODE, notebook="03")

def _stamp(kind: str, value: str) -> None:
    """Print a visible origin tag so no number is mistaken for another kind."""
    print(f"[{kind}] {value}")

MEASURED_TODAY = "measured on THIS GPU in THIS session (physical)"
HISTORICAL   = "frozen reference evidence from the committed campaign"
ESTIMATE     = "analytical estimate or simulation - NOT a measurement"

from lab import student_provenance as prov

def _origin_stamp(category: str) -> str:
    """Stamp text for a provenance category; reuse is NEVER labelled as
    measured-today (fail-closed student provenance contract)."""
    return prov.stamp_text(category)

def gpu_cell(fn):
    """Run a physical exercise only in GPU modes; print why otherwise."""
    if session.mode != "REFERENCE":
        return fn()
    print("skipped physical exercise: COURSE_MODE=REFERENCE (no GPU work launched)")
    return None

def want_own_arms():
    """Whether this session executes its own bounded arms (True unless the
    student opted to reuse the latest physical run to save GPU time)."""
    if session.mode == "REFERENCE" or RUN_OWN_GPU_ARMS:
        return True
    print("RUN_OWN_GPU_ARMS=False: reusing the latest physical campaign run(s) "
          "instead of launching new ones (metered-GPU mode)")
    return False

## 1 · 先说清混淆变量
B0 = 完整 BF16 DiT、40 步。L1 = Turbo BF16 DiT、8 步。*干净的*步数实验应当固定模型、只改变步数（例如同一 Turbo 模型在 8 步与 40 步、相同采样器/调度器下对比）。本实验测的是**可部署配置对**；在任何报告里都要显式区分这两件事。

In [ ]:
CONF0UND_NOTE = ("B0->L1 changes model variant + schedule + steps; "
    "it is NOT a single-variable step-count experiment")
print(CONF0UND_NOTE)
ref = ns.summary_rows()
for k in ("B0", "L1"):
    r = ref[k]
    _stamp(HISTORICAL, f"{k}: median {r['warm_total_median_s']}s n={r['n_measure']}")

## 2 · 在**你的** W7900 上运行两个臂
先跑基线 B0（与你的模式匹配，有界或完整），再跑候选 L1。如果你在本轮M1 已产生 B0，将直接复用并显示其 run_id——记录只追加，不重复浪费算力。

In [ ]:
def run_less_pair():
    prof = "classroom" if session.mode == "GPU_CLASSROOM" else "research"
    b0 = session.run("B0", profile=prof)
    l1 = session.run("L1", profile=prof)
    return b0, l1

pair = gpu_cell(run_less_pair) if want_own_arms() else None
pair_origin = "CURRENT_SESSION_PHYSICAL"
if pair is None:
    b0_id = session.latest_campaign_run("B0")
    l1_id = session.latest_campaign_run("L1")
    if b0_id and l1_id:
        pair = (session.consume(b0_id, why="explicit reuse (shown run_id): B0"),
                session.consume(l1_id, why="explicit reuse (shown run_id): L1"))
        pair_origin = session.consumed[-1].get("category", "COMMITTED_PHYSICAL_REFERENCE")
if pair is not None:
    b0b, l1b = pair
    ab = rt.ab_compare(b0b, l1b, label="B0 (40 steps) vs L1 (Turbo 8 steps)")
    print({k: ab[k] for k in ("classification", "speedup_median",
                              "latency_delta_s", "vram_delta_mb")})

## 3 · 预测加速比 vs 实测加速比
用 M2 流水线分解中的 DiT 占比 `d`（或参考值 0.83）做 Amdahl 预测：predicted = 1/(d·(8/40) + (1−d))。与实测比值对比并解释残差。

In [ ]:
if pair is not None:
    b0b, l1b = pair
    bs, ls = b0b.stats(), l1b.stats()
    d = 0.83  # DiT share of warm latency (replace with YOUR M2 value)
    predicted = 1.0 / (d * (8 / 40) + (1 - d))
    measured = bs["median_s"] / ls["median_s"]
    b0_steps = (bs.get("steps") or 40) or 40
    l1_steps = (ls.get("steps") or 8) or 8
    per_step_b0 = bs["median_s"] / b0_steps
    per_step_l1 = ls["median_s"] / l1_steps
    print(f"predicted (Amdahl, d={d}): {predicted:.2f}x   measured: {measured:.2f}x")
    print(f"per-step: B0 {per_step_b0:.3f}s  L1 {per_step_l1:.3f}s "
          f"(ratio {per_step_b0/per_step_l1:.2f}x)")
    _stamp(_origin_stamp(pair_origin),
           f"measured speedup {measured:.2f}x [{ab['classification']}]")
    _stamp(ESTIMATE, f"Amdahl prediction {predicted:.2f}x (derived from model share)")
    print("limitation: Turbo changes the model too; per-step costs differ; "
          "quality is NOT addressed by this comparison (see section 4)")

## 4 · 质量是另一个维度
少采样不是免费的：Turbo 改变了输出统计特性。课程质量门对 L1 vs B0 采用**距离语义**（LPIPS 是感知*距离*，不是保真度分数），并对成对网格做人工评审。绝不要声称 Turbo 与 40 步基线“等价”。

> **质量判定永远由人完成。** 任何单元都不会自动批准图像质量结论。历史所有者判定**只**适用于其原始评估的部署范围；新的物理运行需要自己的评审（在此之前保持 `PENDING_HUMAN_REVIEW`）。

In [ ]:
qs = ns.summary_rows().get("L1", {})
print("frozen campaign L1 quality verdict:",
      qs.get("quality_verdict", "see evidence/quality/L1/"))
print("paired grids + native-res poster crops: evidence/quality/L1/")
print("in GPU modes run your own gate:  session.run_quality('L1', reference='B0')")
print("your verdict on YOUR images stays PENDING_HUMAN_REVIEW until you review the grid")

## 5 · 动手练习 —— 设计干净实验
设计一个受控的步数实验：固定 Turbo BF16 模型与采样器，步数取 8/16/40，冻结种子与提示词。即便如此，还有什么与 B0 不可比？（模型容量、蒸馏、CFG=1 调度……）

In [ ]:
summary = {
    "mode": session.mode,
    "notebook": session.notebook,
    "executions": [e.get("run_id") or e.get("kind") for e in session.executions],
    "consumed": [c.get("run_id") for c in session.consumed],
    "note": "append-only session record; physical evidence lives under runs/",
}
session.finish(summary)

## 验收标准

- [ ] 你能指出哪些数字是**今天在这块 GPU 上实测的**、哪些是
      **历史参考值**、哪些是**估计/模拟**。
- [ ] 你产生的每次物理运行都在 `runs/` 下有可见的 `run_id`。
- [ ] 你没有把 smoke/课堂样本冒充为正式协议结果。
- [ ] 你能解释你报告的每项测量的一个局限性。

## 复现与贡献

```bash
python -m lab doctor                                  # 准入证据 -> runs/doctor.json
python -m lab run --experiment L1 --date <your-tag>  # 完整研究协议
python -m lab report --scope student --kind all       # 只汇总你自己的 runs
```

提交你的 `runs/<run-id>/*.json` 记录（绝不写 `evidence/`），并在 PR 中说明：
假设 → 协议 → 数字 → 局限。欢迎负结果——测得的退化是科学结果，不是失败。